# 📊 00. KHÁM PHÁ DỮ LIỆU & PHÂN CHIA MASTER STRATIFIED SPLIT (80/10/10)

**Đề tài:** Nhận diện và Phân loại Phát ngôn Độc hại trên Mạng xã hội Tiếng Việt

Notebook này thực hiện:
1. Khám phá và kiểm tra tính toàn vẹn của tập dữ liệu chuẩn vàng `gold_standard_final_48670.csv`.
2. Phân tích phân bố nhãn 2 tầng (`TOXIC/NON-TOXIC` và `EXPLICIT/IMPLICIT`).
3. Thực hiện phân chia Master Stratified Split 80/10/10 (`Train: 38,936`, `Dev: 4,867`, `Test: 4,867`).
4. Phân tích độ dài token thực tế bằng 4 tokenizer trên tập Train và chọn `max_length = 128`.

## 1. Import Thư viện & Cấu hình Đường dẫn

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit
from transformers import AutoTokenizer
import pyvi.ViTokenizer as vi_tokenizer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'gold_standard_final_48670.csv'
SPLITS_DIR = PROJECT_ROOT / 'data_splits'
RESULTS_DIR = PROJECT_ROOT / 'results'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project Root:', PROJECT_ROOT)
print('Data Path:', DATA_PATH)

## 2. Tải & Kiểm tra Tính Toàn vẹn của Tập Dữ liệu Chuẩn vàng

In [ ]:
df = pd.read_csv(DATA_PATH, dtype='string', keep_default_na=False, encoding='utf-8-sig')

print(f'Tổng số dòng: {len(df):,}')
print('Danh sách cột:', list(df.columns))
print(f'Số lượng unique row_id: {df["row_id"].nunique():,}')
print(f'Số bình luận rỗng: {(df["comment"] == "").sum()}')

tox_counts = df['toxicity'].value_counts()
print('\n--- PHÂN BỐ TÁC VỤ 1 (TOXIC / NON-TOXIC) ---')
for k, v in tox_counts.items():
    print(f'  * {k}: {v:,} mẫu ({v/len(df)*100:.2f}%)')

toxic_subset = df[df['toxicity'] == 'TOXIC']
exp_counts = toxic_subset['expression_type'].value_counts()
print('\n--- PHÂN BỐ TÁC VỤ 2 TRÊN TẬP TOXIC (EXPLICIT / IMPLICIT) ---')
for k, v in exp_counts.items():
    print(f'  * {k}: {v:,} mẫu ({v/len(toxic_subset)*100:.2f}%)')

display(df.head())

## 3. Trực quan hóa Phân bố Nhãn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

axes[0].pie(tox_counts.values, labels=tox_counts.index, autopct='%1.2f%%',
            colors=['#81C784', '#E57373'], startangle=140, explode=(0, 0.1),
            textprops={'fontsize': 11, 'weight': 'bold'})
axes[0].set_title('Tác vụ 1: Phân bố Độc hại (N = 48,670)', fontsize=12, fontweight='bold')

axes[1].pie(exp_counts.values, labels=exp_counts.index, autopct='%1.2f%%',
            colors=['#FFB74D', '#BA68C8'], startangle=140, explode=(0, 0.1),
            textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('Tác vụ 2: Kiểu biểu đạt trên tập TOXIC (N = 5,860)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Thực hiện Master Stratified Split 80/10/10

In [ ]:
strata = []
for _, row in df.iterrows():
    if row['toxicity'] == 'NON-TOXIC':
        strata.append('NON_TOXIC')
    elif row['expression_type'] == 'EXPLICIT':
        strata.append('TOXIC_EXPLICIT')
    elif row['expression_type'] == 'IMPLICIT':
        strata.append('TOXIC_IMPLICIT')
    else:
        raise ValueError(f'Unknown stratum: {row}')

df['_stratum'] = strata

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, temp_idx = next(sss1.split(df, df['_stratum']))
train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
dev_rel_idx, test_rel_idx = next(sss2.split(temp_df, temp_df['_stratum']))
dev_df = temp_df.iloc[dev_rel_idx].copy()
test_df = temp_df.iloc[test_rel_idx].copy()

train_df.drop(columns=['_stratum'], inplace=True)
dev_df.drop(columns=['_stratum'], inplace=True)
test_df.drop(columns=['_stratum'], inplace=True)

train_df.to_csv(SPLITS_DIR / 'train.csv', index=False, encoding='utf-8-sig')
dev_df.to_csv(SPLITS_DIR / 'dev.csv', index=False, encoding='utf-8-sig')
test_df.to_csv(SPLITS_DIR / 'test.csv', index=False, encoding='utf-8-sig')

print(f'Train: {len(train_df):,} | Dev: {len(dev_df):,} | Test: {len(test_df):,}')

## 5. Phân tích Phân bố Độ dài Token trên Tập Train

In [ ]:
sys.path.append(str(PROJECT_ROOT / 'src'))
from data import compute_token_length_distribution
len_df = compute_token_length_distribution(SPLITS_DIR / 'train.csv')
display(len_df)